In [8]:
import numpy as np
import pandas as pd

In [124]:
cols = ["date", "time", "epoch", "moteid", "temperature", "humidity", "light", "voltage"]
df = pd.read_csv("temp_sensor.txt", sep=r"\s+", header=None, names=cols)

In [126]:
# Combine date+time into timestamp
df["ts"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
df = df.dropna(subset=["ts", "moteid", "temperature"])

# Intel lab: moteid are the sensor IDs; typically 1..54
df["moteid"] = df["moteid"].astype(int)

# restrict time frame
start = pd.Timestamp("2004-02-28 00:00:00")
end   = pd.Timestamp("2004-03-21 23:59:59")
df = df[(df["ts"] >= start) & (df["ts"] <= end)].copy()

In [127]:
# Resample to 5-min bins per sensor (mean)
df = df.set_index("ts")
temp_bin = (
	df.groupby("moteid")["temperature"]
		.resample('5min')
		.mean()
		.reset_index()
)

In [128]:
min_coverage = 0.7
pivot_raw = temp_bin.pivot(index="ts", columns="moteid", values="temperature").sort_index()
coverage = pivot_raw.notna().mean(axis=0)
sensor_ids = coverage[coverage >= min_coverage].index
pivot = pivot_raw[sensor_ids]

In [129]:
# Fill missing values: time interpolation then ffill/bfill
pivot = pivot.interpolate(limit_direction="both")
pivot = pivot.ffill().bfill()

In [130]:
pivot

moteid,1,2,3,4,6,7,9,10,11,12,...,44,45,46,47,48,49,51,52,53,54
ts,,,,,,,,,,,,,,,,,,,,,
2004-02-28 00:55:00,19.988400,19.567000,20.204000,19.820400,20.4196,19.416733,19.02380,19.317100,18.655600,17.80300,...,17.69275,18.278300,18.356000,18.094200,18.006000,17.48450,18.606600,18.302800,16.127200,15.921400
2004-02-28 01:00:00,19.302400,19.567000,19.424900,19.820400,19.9002,19.416733,19.02380,19.317100,18.655600,17.47715,...,17.69275,18.278300,18.356000,18.094200,18.006000,17.48450,18.201125,17.948600,16.127200,15.921400
2004-02-28 01:05:00,19.163240,19.567000,19.372089,19.820400,19.4200,19.416733,19.02380,18.948200,18.564133,17.15130,...,17.69275,18.278300,18.356000,18.094200,18.006000,17.48450,17.795650,17.594400,16.127200,15.921400
2004-02-28 01:10:00,19.134167,19.537600,19.337925,19.740600,19.4053,19.382025,18.95940,18.889575,18.534733,17.09985,...,17.66384,18.242775,18.295940,18.061067,17.942650,17.48450,17.710600,17.556775,16.006333,15.841911
2004-02-28 01:15:00,19.098560,19.513100,19.327444,19.709644,19.3906,19.339640,18.93000,18.913340,18.505333,17.04840,...,17.57515,18.208720,18.269044,18.026767,17.932360,17.48450,17.656000,17.501650,15.967133,15.799600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2004-03-21 23:35:00,19.861000,20.223600,19.946750,122.153000,122.1530,20.145200,19.10640,24.099500,22.889200,18.22440,...,19.05740,19.622533,76.186500,19.511467,19.650300,19.89040,31.895400,20.044750,19.209300,20.375500
2004-03-21 23:40:00,19.821800,20.194200,19.881008,122.153000,122.1530,20.096200,19.10640,24.290600,22.938200,18.22440,...,19.01820,19.587417,76.635667,19.464100,19.616817,19.86100,33.280467,20.022700,19.179900,20.409800
2004-03-21 23:45:00,19.797300,20.172150,19.815267,122.153000,122.1530,20.086400,19.10640,24.035800,23.026400,18.19500,...,18.98390,19.552300,76.828400,19.442867,19.583333,19.91784,34.394400,19.998200,19.204400,20.380400
